# Bump test recent history

In [ ]:
import asyncio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time, TimeDelta
from lsst.ts.xml.tables.m1m3 import FATable, FAIndex, force_actuator_from_id, actuator_id_to_index
from lsst_efd_client import EfdClient
from lsst.ts.xml.enums.MTM1M3 import BumpTest
from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient, clipDataToEvent, calcNextDay

In [ ]:
def max_error(errors):
    return np.max([np.max(errors), np.max(errors * -1.0)])

def rms_error(times, errors):
    error = 0.0
    num = 0
    for i, t in enumerate(times):
        if (t > pd.to_timedelta(3.0, unit='s') and t < pd.to_timedelta(4.0, unit='s')) or (t > pd.to_timedelta(10.0, unit='s') and t < pd.to_timedelta(11.0, unit='s')):
            num += 1
            error += errors[i]**2
    if num == 0:
        return np.nan
    else:
        return np.sqrt(error / num)
        
async def calc_bumps_and_errors(bump, bt_result, follow):
    BUMP_TEST_DURATION = 14.0  # seconds
    measured_forces_times = []
    following_error_values = []
    t_starts = []
    results = bt_result[bt_result[bump] == BumpTest.TESTINGPOSITIVE]
    for bt_index in range(len(results)):
        t_start = bt_result[bt_result[bump] == BumpTest.TESTINGPOSITIVE].index[bt_index].timestamp()
        t_start = Time(t_start, format='unix_tai', scale="tai")
        #Time(
        #    bt_result[bt_result[bump] == BumpTest.TESTINGPOSITIVE].index[bt_index], #["timestamp"].values[bt_index]- 1.0,
        #    format="unix_tai",
        #    scale="tai",
        #)
        t_starts.append(t_start)#.isot.split('.')[0])
        t_end = Time(
            t_start + TimeDelta(BUMP_TEST_DURATION, format="sec"),
            format="unix_tai",
            scale="tai",
        )
        measured_forces = await client.select_time_series(\
                    "lsst.sal.MTM1M3.forceActuatorData", \
                    [follow, "timestamp"], t_start.utc, t_end.utc)
        
        t0 = measured_forces.index[0]
        measured_forces.index-=t0 #s["timestamp"] -= t0
    
        # It is easier/faster to work with arrays
        measured_forces_time = measured_forces.index.values #["timestamp"].values
        measured_forces_times.append(measured_forces_time)
        following_error_value = measured_forces[follow].values
        following_error_values.append(following_error_value)

    times = []
    max_errors = []
    rms_errors = []
    for i in range(len(measured_forces_times)):
        times.append(t_starts[i])
        max_errors.append(max_error(following_error_values[i]))
        rms_errors.append(rms_error(measured_forces_times[i], following_error_values[i]))
    return [times, rms_errors, max_errors]

In [ ]:
async def actuator_error(client, fa_id, bt_results):    
    # Grab the Force Actuator Data from its ID
    fa_data = force_actuator_from_id(fa_id)
    bt_result = bt_results#[bt_results2["actuatorId"] == fa_id]
    # First the primary forces
    bump = f"primaryTest{fa_data.index}"
    follow = f"primaryCylinderFollowingError{fa_data.index}"
    [ptimes, prms_errors, pmax_errors] = \
        await calc_bumps_and_errors(bump, bt_result, follow)

    # Now the secondary  forces  
    if fa_data.actuator_type.name == "DAA":
        bump = f"secondaryTest{fa_data.s_index}"
        follow = f"secondaryCylinderFollowingError{fa_data.s_index}"
            
        [stimes, srms_errors, smax_errors] = \
            await calc_bumps_and_errors(bump, bt_result, follow)
    else:
        stimes = []; srms_errors = []; smax_errors = []

    return [ptimes, prms_errors, pmax_errors, stimes, srms_errors, smax_errors]


In [ ]:
days_to_plot=30
end = Time.now()#- TimeDelta(300, format='jd')
start = end - TimeDelta(days_to_plot, format='jd')
client = EfdClient('usdf_efd')
bumps = await client.select_time_series(\
                    "lsst.sal.MTM1M3.logevent_forceActuatorBumpTestStatus", \
                    ['*'], start, end)
bumps2 = await client.select_time_series(\
                    "lsst.sal.MTM1M3.logevent_forceActuatorBumpTestStatistics", \
                    ['*'], start, end)
ids = []
for index in range(len(FATable)):
    id = FATable[index].actuator_id
    ids.append(id)
#print(ids)

In [ ]:
begin_time=Time('2025-01-01 16:40:00', format="iso", scale="utc")
end_time=Time('2025-12-31 16:50:00', format="iso", scale="utc")
client = EfdClient("usdf_efd")

df_disable = getEfdData(
    client, "lsst.sal.MTM1M3.command_disableForceActuator", begin=begin_time, end=end_time
)

print(df_disable["actuatorId"])

df_enable= getEfdData(
    client, "lsst.sal.MTM1M3.command_enableForceActuator", begin=begin_time, end=end_time
)
print(df_enable["actuatorId"])

In [ ]:
begin_time=Time('2024-01-01 16:40:00', format="iso", scale="utc")
end_time=Time('2026-04-30 16:50:00', format="iso", scale="utc")
client = EfdClient("usdf_efd")

df_disable = getEfdData(
    client, "lsst.sal.MTM1M3.command_disableForceActuator", begin=begin_time, end=end_time
)

df_enable= getEfdData(
    client, "lsst.sal.MTM1M3.command_enableForceActuator", begin=begin_time, end=end_time
)


In [ ]:
import matplotlib.dates as mdates

In [ ]:
async def get_disabled_actuators(start_time, end_time, efd_name="usdf_efd"):
    """
    Queries the EFD for disabled force actuators and returns data for plotting.
    """
    client = EfdClient(efd_name)
    topic = "lsst.sal.MTM1M3.logevent_enabledForceActuators"

    t1 = Time(start_time, scale='utc')
    t2 = Time(end_time, scale='utc')

    print(f"Querying {topic} from {t1.iso} to {t2.iso}...")

    df = await client.select_time_series(
        topic_name=topic,
        fields=["*"],
        start=t1,
        end=t2
    )
    
    if df.empty:
        print("No events found.")
        return pd.DataFrame()

    # Find forceActuatorEnabled## columns
    enabled_cols = [col for col in df.columns if col.startswith('forceActuatorEnabled')]
    
    if not enabled_cols:
        print("No forceActuatorEnabled## columns found.")
        return pd.DataFrame()

    # Create long-format data for plotting: each disabled event as row
    plot_data = []
    for timestamp, row in df.iterrows():
        for col in enabled_cols:
            if not row[col]:  # Disabled (False)
                actuator_id = int(col.replace('forceActuatorEnabled', ''))
                plot_data.append({'timestamp': timestamp, 'actuator_id': actuator_id})

    if not plot_data:
        print("No disabled actuators found.")
        return pd.DataFrame()

    plot_df = pd.DataFrame(plot_data)
    plot_df.sort_values('timestamp', inplace=True)
    
    print(f"Found {len(plot_df)} disabled actuator events across {len(enabled_cols)} actuators.")
    return plot_df, enabled_cols



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

async def get_disabled_actuators_period(start_time, end_time, efd_name="usdf_efd"):
    client = EfdClient(efd_name)
    topic = "lsst.sal.MTM1M3.logevent_enabledForceActuators"

    t1 = Time(start_time, format="iso", scale="utc")
    t2 = Time(end_time, format="iso", scale="utc")

    df = await client.select_time_series(
        topic_name=topic,
        fields=["*"],
        start=t1,
        end=t2
    )

    if df.empty:
        return pd.DataFrame()

    enabled_cols = [col for col in df.columns if col.startswith("forceActuatorEnabled")]
    plot_data = []

    for timestamp, row in df.iterrows():
        ts = pd.to_datetime(timestamp, utc=True)
        for col in enabled_cols:
            if not row[col]:
                actuator_id = int(col.replace("forceActuatorEnabled", ""))
                plot_data.append({
                    "timestamp": ts,
                    "actuator_id": actuator_id
                })

    return pd.DataFrame(plot_data)

# Define periods
p1_start, p1_end = "2024-10-04 00:00:00", "2025-01-07 23:59:59"
p2_start, p2_end = "2025-03-12 00:00:00", "2025-11-25 23:59:59"
p3_start, p3_end = "2025-11-26 00:00:00", "2026-05-12 23:59:59"

label1 = "2024-10 to 2025-01"
label2 = "2025-03 to 2025-11"
label3 = "2025-11 to 2026-05"

plot1 = await get_disabled_actuators_period(p1_start, p1_end)
plot2 = await get_disabled_actuators_period(p2_start, p2_end)
plot3 = await get_disabled_actuators_period(p3_start, p3_end)

# Colors per period
colors = ["blue", "orange", "green"]
periods = [(plot1, label1), (plot2, label2), (plot3, label3)]

plt.figure(figsize=(16, 8))

for (df, label), color in zip(periods, colors):
    if df.empty:
        continue
    plt.scatter(df["timestamp"], df["actuator_id"], c=color, s=30, alpha=0.7, marker="o", label=label)

plt.xlabel("Timeline (UTC)")
plt.ylabel("Actuator ID")
plt.title("Disabled Force Actuators Over Time")
plt.grid(True, alpha=0.3)

plt.legend()
plt.xticks(rotation=45)
plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=1))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.tight_layout()
plt.show()

# Summary counts per period
def show_counts(df, label):
    print("\n" + "="*70)
    print(label)
    print("="*70)
    if df.empty:
        print("No events.")
    else:
        counts = df["actuator_id"].value_counts().sort_index()
        print(counts)

show_counts(plot1, label1)
show_counts(plot2, label2)
show_counts(plot3, label3)


In [ ]:
async def get_disabled_actuators_periods(start_before, end_before, start_after, end_after, efd_name="usdf_efd"):
    """Get failure data for two periods with monthly breakdown"""
    client = EfdClient(efd_name)
    topic = "lsst.sal.MTM1M3.logevent_enabledForceActuators"
    
    # Query both periods
    t1_before = Time(start_before, scale='utc')
    t2_before = Time(end_before, scale='utc')
    df_before = await client.select_time_series(topic_name=topic, fields=["*"], 
                                              start=t1_before, end=t2_before)
    
    t1_after = Time(start_after, scale='utc')
    t2_after = Time(end_after, scale='utc')
    df_after = await client.select_time_series(topic_name=topic, fields=["*"], 
                                             start=t1_after, end=t2_after)
    
    enabled_cols = [col for col in df_before.columns if col.startswith('forceActuatorEnabled')]
    
    def extract_disabled(df):
        plot_data = []
        for timestamp, row in df.iterrows():
            month = timestamp.strftime('%Y-%m')
            for col in enabled_cols:
                if not row[col]:
                    actuator_id = int(col.replace('forceActuatorEnabled', ''))
                    plot_data.append({'timestamp': timestamp, 'actuator_id': actuator_id, 'month': month})
        return pd.DataFrame(plot_data)
    
    before_df = extract_disabled(df_before)
    after_df = extract_disabled(df_after)
    
    print(f"Before: {len(before_df)} events | After: {len(after_df)} events")
    return before_df, after_df, enabled_cols


In [ ]:
async def get_disabled_actuators_period(start_time, end_time, efd_name="usdf_efd"):
    client = EfdClient(efd_name)
    topic = "lsst.sal.MTM1M3.logevent_enabledForceActuators"

    t1 = Time(start_time, scale='utc')
    t2 = Time(end_time, scale='utc')

    df = await client.select_time_series(
        topic_name=topic,
        fields=["*"],
        start=t1,
        end=t2
    )

    if df.empty:
        return pd.DataFrame()

    enabled_cols = [col for col in df.columns if col.startswith('forceActuatorEnabled')]
    plot_data = []

    for timestamp, row in df.iterrows():
        ts = pd.to_datetime(timestamp, utc=True)
        month = ts.strftime('%Y-%m')
        for col in enabled_cols:
            if not row[col]:
                actuator_id = int(col.replace('forceActuatorEnabled', ''))
                plot_data.append({
                    'timestamp': ts,
                    'actuator_id': actuator_id,
                    'month': month
                })

    return pd.DataFrame(plot_data)


In [ ]:
async def get_disabled_actuators_period(start_time, end_time, efd_name="usdf_efd"):
    client = EfdClient(efd_name)
    topic = "lsst.sal.MTM1M3.logevent_enabledForceActuators"

    t1 = Time(start_time, scale='utc')
    t2 = Time(end_time, scale='utc')

    df = await client.select_time_series(
        topic_name=topic,
        fields=["*"],
        start=t1,
        end=t2
    )

    if df.empty:
        return pd.DataFrame()

    enabled_cols = [col for col in df.columns if col.startswith('forceActuatorEnabled')]
    plot_data = []

    for timestamp, row in df.iterrows():
        ts = pd.to_datetime(timestamp, utc=True)
        month = ts.strftime('%Y-%m')
        for col in enabled_cols:
            if not row[col]:
                actuator_id = int(col.replace('forceActuatorEnabled', ''))
                plot_data.append({
                    'timestamp': ts,
                    'actuator_id': actuator_id,
                    'month': month
                })

    return pd.DataFrame(plot_data)

p1_start, p1_end = '2024-10-04 00:00:00', '2025-01-07 23:59:59'
p2_start, p2_end = '2025-03-12 00:00:00', '2025-11-25 23:59:59'
p3_start, p3_end = '2025-11-26 00:00:00', '2026-05-12 23:59:59'

label1 = '2024-10 to 2025-01'
label2 = '2025-03 to 2025-11'
label3 = '2025-11 to 2026-05'

period1_df = await get_disabled_actuators_period(p1_start, p1_end)
period2_df = await get_disabled_actuators_period(p2_start, p2_end)
period3_df = await get_disabled_actuators_period(p3_start, p3_end)

def safe_count(df):
    return df['actuator_id'].value_counts() if not df.empty else pd.Series(dtype=int)

counts1 = safe_count(period1_df)
counts2 = safe_count(period2_df)
counts3 = safe_count(period3_df)

months1 = pd.period_range('2024-10', '2025-01', freq='M').astype(str).tolist()
months2 = pd.period_range('2025-03', '2025-11', freq='M').astype(str).tolist()
months3 = pd.period_range('2025-11', '2026-05', freq='M').astype(str).tolist()
months_all = pd.period_range('2024-10', '2026-05', freq='M').astype(str).tolist()

def month_series(df, months):
    if df.empty:
        return pd.Series(0, index=months)
    s = df.groupby('month').size()
    return s.reindex(months, fill_value=0)

monthly1 = month_series(period1_df, months1)
monthly2 = month_series(period2_df, months2)
monthly3 = month_series(period3_df, months3)

combined_months = pd.DataFrame({
    label1: monthly1.reindex(months_all, fill_value=0),
    label2: monthly2.reindex(months_all, fill_value=0),
    label3: monthly3.reindex(months_all, fill_value=0),
}, index=months_all)

fig = plt.figure(figsize=(20, 12))

plt.subplot(2, 2, 1)
bins = np.arange(0, 153, 8)
plt.hist(period1_df['actuator_id'], bins=bins, alpha=0.6, label=label1, color='blue')
plt.hist(period2_df['actuator_id'], bins=bins, alpha=0.6, label=label2, color='orange')
plt.hist(period3_df['actuator_id'], bins=bins, alpha=0.6, label=label3, color='green')
plt.xlabel('Actuator ID (0-152)')
plt.ylabel('Failure Events')
plt.title('Total Failures by Actuator ID')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 2)
x = np.arange(len(months_all))
w = 0.28
plt.bar(x - w, combined_months[label1].values, width=w, label=label1, color='blue', alpha=0.7)
plt.bar(x, combined_months[label2].values, width=w, label=label2, color='orange', alpha=0.7)
plt.bar(x + w, combined_months[label3].values, width=w, label=label3, color='green', alpha=0.7)
plt.xticks(x, months_all, rotation=45)
plt.xlim(-0.5, len(months_all) - 0.5)
plt.xlabel('Month')
plt.ylabel('Failure Events')
plt.title('Monthly Failures by Period\n2024-01 to 2026-04')
plt.legend()
plt.grid(True, axis='y', alpha=0.3)

plt.subplot(2, 2, 3)
rate1 = len(period1_df) / len(monthly1) if len(monthly1) else 0
rate2 = len(period2_df) / len(monthly2) if len(monthly2) else 0
rate3 = len(period3_df) / len(monthly3) if len(monthly3) else 0

labels = [label1, label2, label3]
rates = [rate1, rate2, rate3]
colors = ['blue', 'orange', 'green']

plt.bar(labels, rates, color=colors, alpha=0.7, edgecolor='black')
plt.title('Average Monthly Failure Rate')
plt.ylabel('Events / Month')

y_offset = np.max(rates) * 0.02 if np.max(rates) > 0 else 0.1
for i, v in enumerate(rates):
    plt.text(i, v + y_offset, f'{v:.1f}', ha='center', fontweight='bold')
plt.grid(True, axis='y', alpha=0.3)

plt.subplot(2, 2, 4)
top3_p1 = counts1.nlargest(3)
top3_p2 = counts2.nlargest(3)
top3_p3 = counts3.nlargest(3)

x1 = np.arange(3)
x2 = x1 + 3
x3 = x1 + 6

plt.bar(x1, top3_p1.values, 0.8, label=label1, color='blue', alpha=0.8)
plt.bar(x2, top3_p2.values, 0.8, label=label2, color='orange', alpha=0.8)
plt.bar(x3, top3_p3.values, 0.8, label=label3, color='green', alpha=0.8)

xticks = np.concatenate([x1, x2, x3])
xlabels = (
    [f'{label1}-ID{int(i)}' for i in top3_p1.index] +
    [f'{label2}-ID{int(i)}' for i in top3_p2.index] +
    [f'{label3}-ID{int(i)}' for i in top3_p3.index]
)
plt.xticks(xticks, xlabels, rotation=45)
plt.ylabel('Failure Events')
plt.title('Top 3 Worst Actuators by Period')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

def top3_table(counts, label):
    print("\n" + "="*70)
    print(f"TOP 3 WORST ACTUATORS - {label}")
    print("="*70)
    print("Rank | Actuator ID | Failure Events")
    print("-"*40)
    if not counts.empty:
        for rank, (act_id, events) in enumerate(counts.nlargest(3).items(), 1):
            print(f"{rank:3}  |    ID{int(act_id):2}     |     {events:11,}")

top3_table(counts1, label1)
top3_table(counts2, label2)
top3_table(counts3, label3)

print("\nSUMMARY:")
print(f"{label1}: {len(period1_df):,} total events ({rate1:.1f}/month)")
print(f"{label2}: {len(period2_df):,} total events ({rate2:.1f}/month)")
print(f"{label3}: {len(period3_df):,} total events ({rate3:.1f}/month)")
